<a href="https://colab.research.google.com/github/jonhrh1348/aviation-delay/blob/feature%2Fapi_data_setup/00_Initial_setup.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import os
from kafka import KafkaProducer, KafkaConsumer, KafkaAdminClient, TopicPartition
from src.utils.config import *

In [ ]:
# 1. Start the Clickhouse Client for db use
ch_user = os.getenv('CLICKHOUSE_USER')
ch_password = os.getenv('CLICKHOUSE_PW')
ch_host = os.getenv('CLICKHOUSE_HOST')

client = setup_clickhouse_client(ch_user, ch_password, ch_host)
print('Client connected')

In [ ]:
# table schema setup

def create_table_if_not_exists(client, table_name, schema, engine= "MergeTree()"):
    """
    Create a ClickHouse table if it doesn't exist.
    
    Args:
        client: ClickHouse client instance
        table_name: Name of the table to create
        schema: Dictionary mapping column names to their definitions
        engine: Storage engine (default: MergeTree)
    
    Returns:
        bool: True if table was created, False if it already existed
    """
    
    try:
        client.command(f"DROP TABLE IF EXISTS {table_name} SYNC")
    except Exception as e:
        print(f"Warning: Could not drop existing table: {e}")
        raise
    
    # Build column definitions
    columns = ",\n    ".join([f"{col} {defn}" for col, defn in schema.items()])
    
    # Build and execute CREATE TABLE statement
    create_sql = f"""
        CREATE TABLE IF NOT EXISTS {table_name} (
            {columns}
        ) ENGINE = {engine}
        PRIMARY KEY (id);
    """
    
    try:
        client.command(create_sql)
        print(f"Table '{table_name}' created successfully.")

    except Exception as e:
        print(f"Error creating table '{table_name}': {e}")
        raise


In [ ]:
# raw_aviation_data table & raw_hist_weather_data table creation
 
raw_av_schema = {
     'id': 'UUID DEFAULT generateUUIDv4()',
     'insert_time': f"DateTime64(3, 'Asia/Singapore') DEFAULT now64()",
     'type': 'String',
     'status': 'String',
     'departure': 'JSON',
     'arrival': 'JSON',
     'airline': 'Map(String, String)',
     'flight': 'Map(String, String)',
     'codeshared_airline': 'Map(String, String)',
     'codeshared_flight': 'Map(String, String)'
     }
create_table_if_not_exists(client, 'raw_aviation_flights', raw_av_schema)

hist_we_schema = {
    'id': 'UUID DEFAULT generateUUIDv4()',
    'insert_time': f"DateTime64(3, 'Asia/Singapore') DEFAULT now64()",
    'weather_time': 'DateTime',
    'main': 'JSON',
    'wind': 'JSON',
    'clouds': 'JSON',
    'rain': 'JSON',
    'snow': 'JSON',
    'weather': 'Array(JSON)'
}
create_table_if_not_exists(client, 'raw_hist_weather_data', hist_we_schema)